In [ ]:
import glob, fitdecode
import matplotlib.pyplot as plt 
import numpy as np
from fitparse import FitFile, FitParseError
import pandas as pd
from matplotlib.colors import LogNorm
import matplotlib.cm as cm
from datetime import datetime, timedelta
import warnings; [warnings.filterwarnings("ignore", message=msg) for msg in [
    r"invalid field size .* in definition message .* for type uint32.*",
    r"'field \"native_field_num\".*not found in message \"field_description\".*",
    r"'field \"units\".*not found in message \"field_description\".*"
]]

def semicircles_to_degrees(semicircles):
    return semicircles * (180.0 / 2**31)
    
def fit_to_dataframe(fit_file_path):
    fitfile = FitFile(fit_file_path)

    records = []
    for record in fitfile.get_messages('record'):
        data = {}
        for field in record:
            data[field.name] = field.value
        records.append(data)

    return pd.DataFrame(records)


def fit_to_dataframe(filepath):
    with fitdecode.FitReader(filepath) as fit:
        sport_file, flag_is_run = "none", False
        for frame in fit:
            if frame.frame_type == fitdecode.FIT_FRAME_DATA and frame.name == "session":
                flag_is_run = True#frame.get_value("sport") == "running"
                sport_file = frame.get_value("sport")
                # sub_sport_file = frame.get_value("sub_sport")
                break
                
    if flag_is_run:
        records = []
        with fitdecode.FitReader(filepath) as fit:
            print(f"Found running activity: {sport_file}")# - {frame.get_value('sub_sport')}")
            for frame in fit:
                if frame.frame_type == fitdecode.FIT_FRAME_DATA and frame.name == 'record':
                    record_data = {}
                    for field in frame.fields:
                        record_data[field.name] = field.value
                    records.append(record_data)    
            
            return pd.DataFrame(records)
    else:
        print(f"Skipping: {sport_file}")
    
files = glob.glob("./data/coros/*.fit")
print(f"Detected {len(files)} runs")

In [ ]:
fit_to_dataframe(files[0])

In [ ]:
field_names = set()
with fitdecode.FitReader(files[0]) as fit:
    for frame in fit:
        if frame.frame_type == fitdecode.FIT_FRAME_DATA and frame.name == 'record':
            for f in frame.fields:
                field_names.add(f.name)
            if hasattr(frame, "developer_fields"):
                for f in frame.developer_fields:
                    field_names.add(f.name)
print(field_names)


In [ ]:
import fitdecode
import pandas as pd

def dump_fitdecode_to_txt(fit_path, txt_path):
    with open(txt_path, "w", encoding="utf-8") as f:
        with fitdecode.FitReader(fit_path) as fit:
            for frame in fit:
                f.write(f"Frame type: {frame.frame_type}, Name: {getattr(frame, 'name', 'N/A')}\n")
                try:
                    for field in frame.fields:
                        f.write(f"  {field.name}: {field.value}\n")
                except Exception as e:
                    f.write(f"  Field parse error: {e}\n")
                f.write("\n")
def exhaustive_fit_dump(fit_path, txt_path):
    """
    Dump all frames and fields from a FIT file, including partial/corrupted frames.
    Will not fail on invalid field sizes.
    """
    with open(txt_path, "w", encoding="utf-8") as f:
        try:
            with fitdecode.FitReader(fit_path) as fit:
                for frame in fit:
                    # Frame type and name
                    f.write(f"Frame type: {frame.frame_type}, Name: {getattr(frame, 'name', 'N/A')}\n")
                    
                    # Fields
                    if hasattr(frame, "fields"):
                        for field in frame.fields:
                            try:
                                f.write(f"  {field.name}: {field.value}\n")
                            except Exception as e:
                                f.write(f"  Field parse error: {e}\n")
                    
                    # Developer fields
                    if hasattr(frame, "developer_fields"):
                        for dev_field in frame.developer_fields:
                            try:
                                f.write(f"  Dev {dev_field.name}: {dev_field.value}\n")
                            except Exception as e:
                                f.write(f"  Dev field parse error: {e}\n")
                    
                    f.write("\n")
        except Exception as e:
            f.write(f"Top-level parser error: {e}\n")
    print(f"Exhaustive FIT dump written to {txt_path}")
# Example usage:
exhaustive_fit_dump(files[0], "fit_dump.txt")

In [ ]:
dfs = [fit_to_dataframe(f) for f in files]
dfs = [df for df in dfs if df is not None]

In [ ]:
df = dfs[-4]

df

In [ ]:
# --- assuming your dataframe is called df ---
# columns: ['timestamp', 'step_length', 'cadence', 'power', 'speed']

# 1️⃣ Ensure timestamps are datetime and sorted
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# 2️⃣ Smooth raw signals with rolling mean
window = 20  # adjust for your data frequency (e.g. 10–30 samples)
metrics = ['step_length', 'cadence', 'power', 'speed']

for col in metrics:
    df[f'{col}_smooth'] = df[col].rolling(window, center=True).mean()

# 3️⃣ Compute normalized absolute differences (anomaly indicators)
for col in metrics:
    diff = df[col].diff().abs()
    df[f'{col}_anomaly'] = (diff - diff.mean()) / diff.std()

# 4️⃣ Build composite anomaly index
df['anomaly_index'] = df[[f'{c}_anomaly' for c in metrics]].fillna(0).sum(axis=1)
df['anomaly_index_smooth'] = df['anomaly_index'].rolling(window*2, center=True).mean()

# 5️⃣ Create figure with two subplots
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [1, 2]})

# --- (A) Anomaly evolution ---
axes[0].plot(df['timestamp'], df['anomaly_index_smooth'], color='black', lw=2, label='Composite anomaly index')
for col in metrics:
    axes[0].plot(df['timestamp'], df[f'{col}_anomaly'], alpha=0.35, label=f'{col} anomaly')
axes[0].set_title("Behavioral Anomaly Evolution (Composite + Individual Metrics)")
axes[0].set_ylabel("Normalized anomaly intensity")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- (B) Smoothed metrics evolution ---
for j, col in enumerate(metrics):
    if j == 3:
        axes[1].plot(df['timestamp'], df[f'{col}'], lw=1.5, label=f'{col} (smoothed)')
axes[1].set_title("Smoothed Metric Evolution Over Time")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Metric value (normalized scale)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
df

In [ ]:
def semicircles_to_degrees(semicircles):
    return semicircles * (180.0 / 2**31)
def degrees_to_semicircles(degrees):
    return degrees / (180.0 / 2**31)
    
time, pace, heart_rate, lat, lon = [], [], [], [], []
alt, step_len, pow, cad, dist = [], [], [], [], []
alt_smooth, dist_smooth, delta_alt, delta_dist = [], [], [], []
grade, gap, terrain, efficiency, efficiency_adj = [], [], [], [], []
terrain = []

for i, df in enumerate(dfs):
    if ("heart_rate" not in df.columns) or ("position_lat" not in df.columns) or ("power" not in df.columns):
        continue
        
    df = df[~np.isnan(df["speed"])]
    df = df[df["speed"] != 0]
    
    time.append(np.array(df["timestamp"]))
    df["pace"] = 1000 / df["enhanced_speed"] / 60

    pace.append(np.array(df["pace"])); heart_rate.append(np.array(df["heart_rate"])); dist.append(np.array(df["distance"]))
    lat.append(semicircles_to_degrees(np.array(df["position_lat"]))); lon.append(semicircles_to_degrees(np.array(df["position_long"]))) 
    
    alt.append(np.array(df["enhanced_altitude"])); step_len.append(np.array(df["step_length"]));
    pow.append(np.array(df["power"])); cad.append(np.array(df["cadence"]));

    
    df['alt_smooth'] = df['enhanced_altitude'].rolling(window=5, center=True).mean()
    df['dist_smooth'] = df['distance'].rolling(window=5, center=True).mean()
    df['delta_alt'], df['delta_dist'] = df['alt_smooth'].diff(), df['dist_smooth'].diff()
    df['grade'] = 100 * (df['delta_alt'] / df['delta_dist'])
    df['gap'] = df["pace"] * (1 + df['grade'] / 10)

    grade.append(df["grade"]); gap.append(df["gap"])
    alt_smooth.append(df["alt_smooth"]); dist_smooth.append(df["dist_smooth"])
    delta_alt.append(df["delta_alt"]); delta_dist.append(df["delta_dist"])
    
    df['terrain'] = pd.cut(df['grade'], bins=[-np.inf, -3, 3, np.inf], labels=[-1, 0, 1])
    df.groupby('terrain', observed=True)[['pace', 'heart_rate', 'power']].mean()
    
    df['efficiency'] = df['enhanced_speed'] / df['power']
    df['efficiency_adj'] = df['efficiency'] / (1 + df['grade'].abs() / 10)

    terrain.append(np.array(df["terrain"]))

In [ ]:
for i in range(len(lon)):
    fig, ax = plt.subplots(figsize=(6, 5))


    # plt.plot(lon[i], lat[i], color="darkblue", ls="-", lw=2)
    from matplotlib.colors import LinearSegmentedColormap

    # Define a custom colormap for altitude
    colors = [
        (0.0, "#f5e663"),   # yellow
        (0.33, "#f28e2b"),  # orange
        (0.66, "#e63946"),   # red
        (1.0, "#8b4513")
    ]

    altitude_cmap = LinearSegmentedColormap.from_list("altitude_cmap", colors)

    plt.scatter(lon[i], lat[i], c=alt[i], marker=".", cmap=altitude_cmap)

    ax.set_axis_off()
    fig.patch.set_alpha(0)        # Transparent figure background
    ax.patch.set_alpha(0)         # Transparent axes background

    # Save as PNG with transparent background
    plt.savefig("trail_path.png", dpi=300, bbox_inches='tight', pad_inches=0, transparent=True)
    plt.show()

In [ ]:
import contextily as cx
import geopandas
import rasterio
from rasterio.plot import show as rioshow
import matplotlib.pyplot as plt
from geodatasets import get_path

In [ ]:
data_url = "https://ndownloader.figshare.com/files/20232174"
db = geopandas.read_file(data_url)

In [ ]:
nightlights = cx.providers.NASAGIBS.ViirsEarthAtNight2012
ireland = cx.Place("Spain", source=nightlights)

In [ ]:
import xyzservices.providers as xyz
k = 0
delta = 0.02
min_lat, max_lat = np.nanmean(lat[k]) - delta, np.nanmean(lat[k]) + delta
min_lon, max_lon = np.nanmean(lon[k]) - delta, np.nanmean(lon[k]) + delta


extent = [min_lon, max_lon, min_lat, max_lat]

fig, ax = plt.subplots(figsize=(9, 9))

ghent_img, ghent_ext = cx.bounds2img(
    min_lon, min_lat, max_lon, max_lat, ll=True,source=xyz.OpenTopoMap
    # OpenTopoMap, CyclOSM, Esri.WorldTopoMap, Esri.WorldImagery, Esri.WorldShadedRelief
)

import geopandas as gpd
from shapely.geometry import LineString
ax.imshow(ghent_img, extent=[min_lon, max_lon, min_lat, max_lat], aspect="auto")

line = LineString(zip(degrees_to_semicircles(lon[k]), degrees_to_semicircles(lat[k])))
gdf = gpd.GeoDataFrame(geometry=[line], crs="EPSG:4326").to_crs(epsg=3857)
gdf.plot(ax=ax, color="red", linestyle="--")


# for k in [k]:#range(len(lon)):
#     plt.plot(lon[k], lat[k], color="r", ls="--")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot map
ax.imshow(ghent_img, extent=ghent_ext, interpolation='bilinear')

# Plot line
gdf.plot(ax=ax, linewidth=2, color='red')

# Ensure extent is set
ax.set_xlim(ghent_ext[0], ghent_ext[2])
ax.set_ylim(ghent_ext[1], ghent_ext[3])

plt.show()

In [ ]:
k = 0

delta = 0.02
min_lat, max_lat = np.nanmean(lat[k]) - delta, np.nanmean(lat[k]) + delta
min_lon, max_lon = np.nanmean(lon[k]) - delta, np.nanmean(lon[k]) + delta


ghent_img, ghent_ext = cx.bounds2img(min_lon, min_lat, max_lon, max_lat,
                                     ll=True,
                                     source=xyz.NASAGIBS.BlueMarble
                                    )


f, ax = plt.subplots(1, figsize=(8, 15))
ax.imshow(ghent_img, extent=[min_lon, max_lon, min_lat, max_lat], aspect="auto")

for k in [k]:#range(len(lon)):
    plt.plot(lon[k], lat[k])

In [ ]:
k = 0
from scipy.ndimage import gaussian_filter1d

terrain_colors = {-1: "lightblue", 0: "0.8", 1: "lightcoral"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

for ax in [ax1, ax2]:
    start_idx = 0
    terrain_smooth = gaussian_filter1d(terrain[k].astype(float), sigma=10)
    for i in range(1, len(time[k])):
        if terrain_smooth[i] != terrain_smooth[i-1] and not np.isnan(terrain_smooth[i-1]):
            start_time = dist[k][start_idx]
            end_time = dist[k][i-1]
    
            ax.axvspan(start_time, end_time,
                        color=terrain_colors[np.round(terrain_smooth[i-1])],
                        alpha=1,)  # semi-transparent
    
            start_idx = i
        
ax2.plot(dist[k], gaussian_filter1d(alt_smooth[k], 3), "-k")
ax1.plot(dist[k], pace[k], "-k")


ax2.fill_between(dist[k], gaussian_filter1d(alt_smooth[k], 3), color="0.35", alpha=0.3)

ax1.set_ylim(10, 3)
for ax in [ax1, ax2]:
    ax2.set_ylim(np.nanmin(alt[k])-5 if np.nanmin(alt[k]) >= 0.0 else -5)
    ax2.set_xlim(np.nanmin(dist[k]), np.nanmax(dist[k]))
plt.show()

In [ ]:
for k in [0]:
    plt.plot(lon[k], lat[k])

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

delta = 0.02
min_lat, max_lat = np.nanmean(lat[k]) - delta, np.nanmean(lat[k]) + delta
min_lon, max_lon = np.nanmean(lon[k]) - delta, np.nanmean(lon[k]) + delta

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([min_lon, max_lon, min_lat, max_lat])

for k in [0]:
    plt.plot(lon[k], lat[k])

# Add land with natural earth shading
ax.add_feature(cfeature.LAND.with_scale('10m'), facecolor='lightgreen')

# Add ocean with color
ax.add_feature(cfeature.OCEAN.with_scale('10m'), facecolor='lightblue')

# Add lakes and rivers
ax.add_feature(cfeature.LAKES.with_scale('10m'), facecolor='blue', alpha=0.5)
ax.add_feature(cfeature.RIVERS.with_scale('10m'), edgecolor='blue')

# Add coastlines with high resolution
ax.coastlines(resolution='10m')

# Optionally add gridlines
gl = ax.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False

plt.show()


In [ ]:
plt.hist2d(df['pace'], df['heart_rate'], bins=50, cmap='plasma')
plt.colorbar(label='Count')
plt.title('Heart Rate vs Pace')
plt.xlabel('Pace (min/km)')
plt.ylabel('Heart Rate (bpm)')
plt.show()

In [ ]:
plt.scatter(df['distance'], df['altitude'], c=df['pace'], cmap='coolwarm', s=3)
plt.colorbar(label='Pace (min/km)')
plt.title('Elevation Profile')
plt.xlabel('Distance (m)')
plt.ylabel('Altitude (m)')
plt.show()


In [ ]:
mask = ~np.isnan(df["grade"])
plt.hist2d(
    df['grade'][mask], 
    df['power'][mask], 
    bins=50, cmap='viridis')
plt.colorbar(label='Count')
plt.title('Power vs Grade')
plt.xlabel('Grade (%)')
plt.ylabel('Power (W)')
plt.show()


In [ ]:
df['pace_roll'] = df['pace'].rolling(10).mean()
df['hr_roll'] = df['heart_rate'].rolling(10).mean()

plt.plot(df['timestamp'], df['pace_roll'], label='Pace (rolling)')
plt.plot(df['timestamp'], df['hr_roll'], label='Heart Rate (rolling)')
plt.title('Rolling Pace and Heart Rate')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.show()


In [ ]:
plt.scatter(df['cadence'], df['step_length'], c=df['power'], cmap='inferno', s=5)
plt.colorbar(label='Power (W)')
plt.title('Cadence vs Step Length colored by Power')
plt.xlabel('Cadence (spm)')
plt.ylabel('Step Length (m)')
plt.show()


In [ ]:
plt.scatter(lon, lat, c=df['altitude'], cmap='terrain', s=2)
plt.colorbar(label='Altitude (m)')
plt.title('Trail Route by Elevation')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axis('equal')
plt.show()


In [ ]:
plt.hist(df['pace'], bins=40, color='skyblue', edgecolor='black')
plt.title('Distribution of Pace')
plt.xlabel('Pace (min/km)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
hr_zones = [100, 120, 140, 160, 180, 200]
zone_labels = [f"{hr_zones[i]}–{hr_zones[i+1]}" for i in range(len(hr_zones)-1)]

zone_counts = pd.cut(df['heart_rate'], bins=hr_zones).value_counts().sort_index()
plt.bar(zone_labels, zone_counts)
plt.title('Time in Heart Rate Zones')
plt.xlabel('HR Zone (bpm)')
plt.ylabel('Time Points')
plt.xticks(rotation=45)
plt.show()


In [ ]:
time_norm = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds()
time_norm = time_norm / time_norm.max()

colors = cm.inferno(time_norm)

plt.figure(figsize=(8, 6))
plt.scatter(lon, lat, c=colors, s=2)
plt.title('Trail Path Colored by Time')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axis('equal')
plt.show()

In [ ]:
plt.figure(figsize=(12, 2))
plt.scatter(df['timestamp'], df['altitude'], c=df['power'], cmap='hot', s=2)
plt.colorbar(label='Power (W)')
plt.title('Power Flow Over Elevation')
plt.xlabel('Time')
plt.ylabel('Altitude')
plt.tight_layout()
plt.show()


In [ ]:
alt = df['altitude'].rolling(10).mean()
plt.fill_between(df['distance'], alt, color='sienna', alpha=0.3)
plt.plot(df['distance'], alt, color='black', linewidth=1)
plt.title('Shaded Elevation Profile')
plt.xlabel('Distance (m)')
plt.ylabel('Altitude (m)')
plt.show()


In [ ]:
from matplotlib.collections import LineCollection

mask = ~np.isnan(df["grade"])

x = df['distance'][mask].values
y = df['altitude'][mask].values
points = np.array([x, y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

colors = df['grade'].clip(-30, 30)  # clip extreme values
norm = plt.Normalize(-30, 30)
lc = LineCollection(segments, cmap='coolwarm', norm=norm)
lc.set_array(colors)
lc.set_linewidth(2)

fig, ax = plt.subplots()
line = ax.add_collection(lc)
plt.colorbar(line, label='Grade (%)')
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
plt.title('Gradient-Encoded Elevation Profile')
plt.xlabel('Distance (m)')
plt.ylabel('Altitude (m)')
plt.show()


In [ ]:
import numpy as np

times = df['timestamp'].dt.time
hours = df['timestamp'].dt.hour + df['timestamp'].dt.minute / 60
radii = df['heart_rate']

theta = (hours / 24) * 2 * np.pi  # 24-hour circle

plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
ax.scatter(theta, radii, c=radii, cmap='cool', s=3)
ax.set_theta_direction(-1)
ax.set_theta_offset(np.pi / 2)
plt.title("Heart Rate Over 24-Hour Clock", pad=20)
plt.show()


In [ ]:
plt.scatter(df['pace'], df['power'], c=df['grade'], cmap='bwr', s=5)
plt.xlabel('Pace (min/km)')
plt.ylabel('Power (W)')
plt.title('Power vs Pace (Color = Grade)')
plt.colorbar(label='Grade (%)')
plt.grid(True)
plt.show()


In [ ]:
plt.scatter(grade, pace)
plt.

In [ ]:
avg_metrics = {
    'Pace': df['pace'].mean(),
    'Heart Rate': df['heart_rate'].mean(),
    'Cadence': df['cadence'].mean(),
    'Power': df['power'].mean(),
    'Step Length': df['step_length'].mean(),
    'Vert Rate': df['altitude'].diff().fillna(0).mean() / (df['timestamp'].diff().dt.total_seconds().fillna(1).mean() / 60)
}

labels = list(avg_metrics.keys())
values = list(avg_metrics.values())

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
values += values[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, values, 'o-', linewidth=2)
ax.fill(angles, values, alpha=0.3)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
plt.title("Trail Run Fitness Fingerprint")
plt.show()


In [ ]:
plt.scatter(df["position_lat"], df["position_long"], c=df["heart_rate"], alpha=0.1)

In [ ]:
plt.plot(df[""])

In [ ]:
# Print the number of tracks
print("Number of tracks:", len(gpx.tracks))

# Get the first track
track = gpx.tracks[0]
print("Track object:", track)

# Print track type and name
print("Track type:", track.type)
print("Track name:", track.name)

# Print the segments
print("Track segments:", track.segments)

In [ ]:
# Get the first segment of the track
segment = track.segments[0]

# Print number of points in the segment
print("Number of points in segment:", len(segment.points))

# Get a random point (e.g., the 45th point, index 44)
random_point = segment.points[44]
print("Random point object:", random_point)

# Print latitude, longitude, elevation, and time of the point
print("Latitude:", random_point.latitude)
print("Longitude:", random_point.longitude)
print("Elevation:", random_point.elevation)
print("Time:", random_point.time)


In [ ]:
# Access extensions of the random point
print("Extensions:", random_point.extensions)

# Get the first extension element
tpe = random_point.extensions[0]

# Print each child tag and its text content
print("TrackPointExtension contents:")
for child in tpe:
    print(child.tag, child.text)


In [ ]:
print(segment.length_2d(), segment.length_3d())

In [ ]:
    with open(file) as f:
        gpx = gpxpy.parse(f)

In [ ]:
gpx.tracks[0].segments[0].points

In [ ]:
segment.points

In [ ]:
parse_gpx.get_gpx_point_data(segment.points[0])

In [ ]:
df = parse_gpx.get_dataframe_from_gpx(file)
df

In [ ]:
plt.plot(df["heart_rate"])

In [ ]:
plt.scatter(df["latitude"], df["longitude"], "-")